In [ ]:
import zipfile
import os

zip_file_path = '/content/archive (3).zip'
output_dir = '/content/unzipped_data'

os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print(f'Files unzipped to: {output_dir}')
print('Contents of unzipped directory:')
for root, dirs, files in os.walk(output_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Files unzipped to: /content/unzipped_data
Contents of unzipped directory:
/content/unzipped_data/CWC2023.csv


In [ ]:
import pandas as pd

file_path = '/content/unzipped_data/CWC2023.csv'
df = pd.read_csv(file_path)

print('First 5 rows of the dataset:')
display(df.head())

print('\nDataset information:')
df.info()

First 5 rows of the dataset:


,Match ID,Match Date,Match Time,City,Stadium,Team A,Team B,Toss Winner,Toss Decision,Score A,...,No Balls B,Penalty B,Extras B,Wining Team,Margin,Man of the Match,Umpire 1,Umpire 2,TV Umpire,Match Refree
0,1,05-10-2023,2:00 PM,Ahmedabad,Narendra Modi Stadium,England,NewZealand,NewZealand,Field,282,...,0,0,6,NewZealand,9 Wickets,Rachin Ravindra,Kumar Dharmasena,Nitin Menon,Paul Wilson,Javagal Srinath
1,2,06-10-2023,2:00 PM,Hyderabad,Eden Gardens,Pakistan,Netherlands,Netherlands,Field,286,...,1,0,9,Pakistan,81 Runs,Saud Shakeel,Adrian Holdstock,Chris Brown,Rod Tucker,Jeff Crowe
2,3,07-10-2023,2:00 PM,Dharamshala,HPCA Stadium,Afghanistan,Bangladesh,Bangladesh,Field,156,...,0,0,8,Bangladesh,6 Wickets,Mehidy Hasan Miraz,Joel Wilson,Kumar Dharmasena,Ahsan Raza,Andy Pycroft
3,4,07-10-2023,2:00 PM,Delhi,Arun Jaitley Stadium,South Africa,Sri Lanka,Sri Lanka,Field,428,...,1,0,23,South Africa,102 Runs,Aiden Markram,Richard Illingworth,Sharfuddoula,Michael Gough,Javagal Srinath
4,5,08-10-2023,2:00 PM,Chennai,M. A. Chidambaram Stadium,Australia,India,Australia,Bat,199,...,0,0,12,India,6 wickets,KL Rahul,Chris Gaffaney,Richard Kettleborough,Chris Brown,Richie Richardson



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 44 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Match ID          48 non-null     int64  
 1   Match Date        48 non-null     object 
 2   Match Time        48 non-null     object 
 3   City              48 non-null     object 
 4   Stadium           48 non-null     object 
 5   Team A            48 non-null     object 
 6   Team B            48 non-null     object 
 7   Toss Winner       48 non-null     object 
 8   Toss Decision     48 non-null     object 
 9   Score A           48 non-null     int64  
 10  Wickets A         48 non-null     int64  
 11  Overs Played A    48 non-null     float64
 12  Maiden Overs A    48 non-null     int64  
 13  4s A              48 non-null     int64  
 14  6s A              48 non-null     int64  
 15  Boundaries A      48 non-null     int64  
 16  Runrate A         48 non

In [ ]:
# Identify the target variable
target = 'Wining Team'

# Investigate 'Runrate B' column
print("Unique values in 'Runrate B':", df['Runrate B'].unique())

# Attempt to convert 'Runrate B' to numeric, coercing errors to NaN
df['Runrate B'] = pd.to_numeric(df['Runrate B'], errors='coerce')

# Fill any NaN values that resulted from coercion (e.g., with the mean or 0)
df['Runrate B'] = df['Runrate B'].fillna(df['Runrate B'].mean())

# Select numerical features for KNN. Exclude 'Match ID' as it's an identifier.
# Also exclude other object columns for now, except the target.
feature_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
feature_columns.remove('Match ID') # Remove identifier column

features = df[feature_columns]
labels = df[target]

print(f"\nSelected features: {feature_columns}")
print(f"Target variable: {target}")

# Check for any remaining non-numeric columns in features
print("\nData types of selected features after cleaning:")
print(features.dtypes)

# Import necessary libraries for model training and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42)

print(f"\nTraining set shape (X, y): {X_train.shape}, {y_train.shape}")
print(f"Test set shape (X, y): {X_test.shape}, {y_test.shape}")

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully.")

# Initialize and train the KNN model
knn = KNeighborsClassifier(n_neighbors=5) # Using 5 neighbors as a common starting point
knn.fit(X_train_scaled, y_train)

print("\nKNN model trained successfully.")

# Make predictions on the scaled test set
y_pred = knn.predict(X_test_scaled)

# Evaluate the model
print("\nModel Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Unique values in 'Runrate B': ['7.78' '5' '4.55' '7.27' '4.86' '4.79' '4.69' '7.13' '7.8' '4.33' '5.78'
 '6.29' '5.3' '6.08' '4.83' '4' '6.28' '6.7' '5.44' '7.72' '5.7' '5.83'
 '4.99' '4.28' '6.23' '5.22' '7.66' '4.58' '3.7' '5.33' '4.51' '4.7'
 '2.79' '3.84' '7.84' '5.25' '3.05' '6.85' '5.82' '6.78' '7.37' '5,20'
 '6.87' '5.6' '6.69' '4.26']

Selected features: ['Score A', 'Wickets A', 'Overs Played A', 'Maiden Overs A', '4s A', '6s A', 'Boundaries A', 'Runrate A', 'Leg Byes A', 'Byes A', 'Wides A', 'No Balls A', 'Penalty A', 'Extras A', 'Score B', 'Wickets B', 'Overs Played B', 'Maiden Overs B', '4s B', '6s B', 'Boundaries B', 'Runrate B', 'Leg Byes B', 'Byes B', 'Wides B', 'No Balls B', 'Penalty B', 'Extras B']
Target variable: Wining Team

Data types of selected features after cleaning:
Score A             int64
Wickets A           int64
Overs Played A    float64
Maiden Overs A      int64
4s A                int64
6s A                int64
Boundaries A        int64
Runrate A       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Task
Analyze the current poor performance of the KNN model, including potential causes such as dataset size and class imbalance, and then suggest actionable next steps for improving the model's accuracy and overall performance.

## Analyze and Interpret KNN Results

### Subtask:
Interpret the low accuracy and other metrics from the KNN model evaluation. Identify potential causes like dataset size and class imbalance.


## Interpret KNN Results

Based on the 'Model Evaluation' output from the previous cell, let's interpret the results:

1.  **Overall Accuracy**: The model achieved an accuracy of `0.13` (13%). This is extremely low, indicating that the model is performing only slightly better than random guessing for a multi-class classification problem with 9 classes in the test set. For context, if there were 9 classes and the model guessed randomly, it would have an accuracy of approximately 1/9, or around 11%. So, 13% is barely above that.

2.  **Classification Report Analysis**:
    *   **Precision, Recall, F1-score**: Many classes have `0.00` for precision, recall, and f1-score. This means the model completely failed to correctly predict any instances of these classes. For example, 'Afghanistan', 'Bangladesh', 'England', 'India', 'Netherlands', 'NewZealand', and 'Sri Lanka' all show 0.00 across these metrics. This implies the model never predicted these teams as winners, or if it did, those predictions were incorrect.
    *   **Australia**: Has a recall of `1.00`, meaning all actual 'Australia' wins in the test set were correctly identified. However, its precision is `0.10`, indicating that only 10% of the times the model predicted 'Australia' to win, it was actually correct. This suggests the model is heavily biased towards predicting 'Australia'.
    *   **South Africa**: Shows a precision of `1.00` and recall of `0.33`. This means that when the model *did* predict 'South Africa' as the winner, it was always correct (high precision). However, it only caught 33% of the actual 'South Africa' wins (low recall).
    *   **Support**: This column is crucial. Notice that many classes have very low support in the test set (e.g., 'Australia', 'Bangladesh', 'Netherlands', 'NewZealand', 'Sri Lanka' all have only 1 sample each, while 'Afghanistan', 'India' have 2, and 'England', 'South Africa' have 3). This highlights a severe **class imbalance** and a very **small dataset size** for the test set. With so few examples per class, it's very difficult for any model, especially KNN which relies on distances, to learn meaningful patterns and make accurate predictions.
    *   **`UndefinedMetricWarning`**: The warning messages confirm that precision is ill-defined for classes with no predicted samples, further underscoring the model's inability to predict several classes.

3.  **Confusion Matrix Analysis**:
    *   The confusion matrix visually confirms the issues seen in the classification report. Most of the predictions are concentrated in one or two classes. For example, the row for 'Australia' shows that the model predicted 'Australia' correctly once (diagonal element `[1,1]`), but also misclassified other teams as 'Australia' (e.g., `[0,1]`, `[1,1]`, `[2,1]`, `[3,1]`, `[4,1]`, `[5,1]`, `[6,1]`, `[7,1]`, `[8,1]` where many actual classes are predicted as Australia). Specifically, the column corresponding to 'Australia' (the second column) has a large sum, indicating that 'Australia' is the most frequently predicted class, regardless of the true label.
    *   There are many zeros across the diagonal for most classes, meaning the model failed to correctly predict most of the winning teams.

4.  **Dataset Size**: The training set (`X_train.shape`) has `33` samples and the test set (`X_test.shape`) has `15` samples. With 10 distinct winning teams in the entire dataset (and 9 in the test set), this dataset is extremely small for a multi-class classification problem. Each class has, on average, only a handful of examples, which is insufficient for the model to learn robust decision boundaries.

5.  **Class Imbalance**: As observed in the 'support' column of the classification report, there's a significant class imbalance. Some classes have as little as one sample in the test set, making it nearly impossible for the model to generalize. The `y_train` and `y_test` series also show this imbalance, with some teams having many wins and others very few.

**Summary of Observations and Potential Causes**:

The KNN model's performance is very poor, with an accuracy of only 13%. This can be attributed primarily to two major factors:

*   **Small Dataset Size**: With only 48 total matches and 15 in the test set, the model does not have enough data to learn the complex relationships between features and the winning team.
*   **Severe Class Imbalance**: The number of examples for each winning team varies greatly, with many classes having critically few samples in both the training and test sets. This makes the model biased towards the majority classes (if any, in this case 'Australia' seems to be over-predicted) and unable to learn about the minority classes. The `UndefinedMetricWarning` is a direct consequence of this.

To improve the model, addressing these data limitations (e.g., collecting more data, using resampling techniques for imbalance, or employing more robust models suitable for small, imbalanced datasets) would be crucial.

## Suggest Improvements for Model Performance

### Subtask:
Propose concrete next steps to improve the model's accuracy, such as cross-validation, hyperparameter tuning, feature engineering, or trying alternative models.


## Summary:

### Data Analysis Key Findings

*   **Extremely Low Accuracy**: The KNN model achieved an accuracy of only 0.13 (13%), which is marginally better than random guessing for a 9-class classification problem (approximately 11%).
*   **Poor Performance for Most Classes**: Many classes, including 'Afghanistan', 'Bangladesh', 'England', 'India', 'Netherlands', 'NewZealand', and 'Sri Lanka', exhibited precision, recall, and F1-scores of $0.00$. This indicates the model completely failed to correctly predict any instances of these teams.
*   **Prediction Bias Towards 'Australia'**: The model showed a strong bias by correctly identifying all actual 'Australia' wins (recall of $1.00$) but had a low precision of $0.10$ for 'Australia', meaning only 10% of its 'Australia' predictions were correct. This suggests over-prediction of 'Australia'.
*   **Severe Class Imbalance**: The dataset displayed significant class imbalance, with many classes having very low support (1-3 samples) in the test set, leading to `UndefinedMetricWarning` for various metrics in the classification report.
*   **Insufficient Dataset Size**: The training set comprised only 33 samples, and the test set had 15 samples. This extremely limited data, distributed across 9-10 distinct classes, is inadequate for a robust multi-class classification model.

### Insights or Next Steps

*   The current model's poor performance is primarily attributable to the critically small dataset size and severe class imbalance. Any attempts to improve model accuracy must first address these fundamental data limitations.
*   Future efforts should focus on strategies to mitigate data scarcity and imbalance, such as collecting more data, utilizing data augmentation techniques, employing resampling methods (e.g., SMOTE), or exploring models specifically designed for small and imbalanced datasets.
